# AMEX Enterprise Credit Risk Platform
## Notebook 37 -- Early Payment Default: Financial-Impact Reporting & Packaging
### Phase 2 . Problem Statement 5: Early Payment Default Detection

CRISP-DM stage: **Evaluation & Deployment (reporting)**. Sprint 1, Notebook 4 of 4 for this problem -- the final notebook of Problem 5. Depends on Problem 1 Notebooks 02/05/08 (real default rate, champion model name, and the real, inherited EAD/LGD assumptions) and this problem's own Notebooks 34-36 (policy, modeling results, validation & deployment outcome). Reuses the exact reporting/packaging template proven in Notebook 29 (Problem 4) and reused again by Problem 3's Notebook 33.

**What this notebook does (real, computed on your machine when you run it):**
- States Problem 5's own financial narrative -- **Early-Warning Value**: how many months earlier a customer can be flagged (median full-history statement count vs. the winning early window K), and how many real defaulters are captured through a top-4%-highest-risk flag list built from Notebook 35/36's own measured top-4% capture rate
- Inherits EAD and LGD from Problem 1's Notebook 08 rather than re-guessing them, and clearly labels the one estimate that is DERIVED rather than directly measured (the holdout defaulter count, since Notebooks 35/36 do not persist per-customer holdout labels)
- Computes real loss-prevention opportunity, Year-1 ROI, and payback period from an explicit, fully editable set of financial assumptions
- Writes SMART suggestions for six organizational levels, from frontline early-warning analysts up to the CFO, each referencing real numbers from this problem's own notebooks (including Notebook 36's bootstrap CI, calibration gap, and PSI)
- Produces a Word report, a formula-linked Excel workbook (editing an assumption recalculates every downstream dollar figure), and an interactive HTML dashboard
- Handles the honest edge case plainly: if the estimated annual benefit is ever $0 (e.g. very small holdout populations), ROI and payback are reported as "N/A -- no measurable benefit under current assumptions" everywhere, rather than crashing or fabricating a number

**What this notebook does NOT do:** train or validate any model -- that's Notebooks 35 and 36. This is reporting and packaging only.

Zero-fabrication: every number in this report is computed live from this run's real upstream notebook outputs; only the assumptions explicitly labeled ASSUMPTION are illustrative and meant to be edited to your institution's real figures.

**This is the final notebook of Problem 5 -- once it completes, Phase 2 (Problems 3, 4, and 5) is complete.**


In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOKS 34/35/36's REAL OUTPUTS
# =============================================================================
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebooks 34/35/36's Real Outputs")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P1_ROOT = PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction"
P5_ROOT = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "Problem5_Early_Payment_Default_Detection"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

PILLAR_DIRS = {
    "p5_policy": P5_ROOT / "policy",
    "p5_modeling": P5_ROOT / "modeling",
    "p5_deployment": P5_ROOT / "deployment",
    "p5_reporting_packaging": P5_ROOT / "05_Financial_Impact_Reporting_Packaging",
}
for _d in PILLAR_DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

P1_CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB08_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_08_summary.json"
NB34_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_34_summary.json"
NB35_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_35_summary.json"
NB36_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_36_summary.json"

for _p, _fix in [
    (P1_CONFIG_PATH, "run Problem 1's Notebook 01 first."),
    (NB02_SUMMARY_PATH, "run Problem 1's Notebook 02 first."),
    (NB05_SUMMARY_PATH, "run Problem 1's Notebook 05 first."),
    (NB08_SUMMARY_PATH, "run Problem 1's Notebook 08 first (this notebook inherits its real "
                         "EAD/LGD assumptions rather than re-guessing them)."),
    (NB34_SUMMARY_PATH, "run 34_early_payment_default_business_understanding.ipynb first."),
    (NB35_SUMMARY_PATH, "run 35_early_payment_default_modeling.ipynb first."),
    (NB36_SUMMARY_PATH, "run 36_early_payment_default_validation_deployment.ipynb first."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p.name} not found in its expected location.\nFix: {_fix}")

with open(P1_CONFIG_PATH, "r", encoding="utf-8") as f:
    P1_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB08_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB08_SUMMARY = json.load(f)
with open(NB34_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB34_SUMMARY = json.load(f)
with open(NB35_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB35_SUMMARY = json.load(f)
with open(NB36_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB36_SUMMARY = json.load(f)

POLICY_PATH = Path(NB34_SUMMARY["policy_path"])
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    EARLY_DEFAULT_POLICY = json.load(f)
STATEMENT_COUNT_STATS = EARLY_DEFAULT_POLICY["statement_count_stats"]

MODELING_RESULTS_PATH = Path(NB35_SUMMARY["modeling_results_path"])
with open(MODELING_RESULTS_PATH, "r", encoding="utf-8") as f:
    MODELING_ARTIFACT = json.load(f)

WINNING_K = NB36_SUMMARY["winning_k"]
MEETS_KPI = NB36_SUMMARY["meets_kpi_target"]
RECOMMENDED_FOR_PRODUCTION = NB36_SUMMARY["recommended_for_production"]
WINNING_K_RESULT = MODELING_ARTIFACT["results_by_k"][str(WINNING_K)]

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
FULL_HISTORY_AUC = MODELING_ARTIFACT["full_history_reference_auc"]
LIVE_DEFAULT_RATE = NB02_SUMMARY["join_validation"]["live_default_rate"]
N_HOLDOUT = WINNING_K_RESULT["holdout_customers"]
EAD_PER_ACCOUNT_USD = NB08_SUMMARY["ead_per_account_usd_assumption"]
LGD_ASSUMPTION = NB08_SUMMARY["lgd_assumption"]

print(f"Winning early window (Notebook 36)      : K={WINNING_K}")
print(f"Meets KPI target / recommended for prod  : {MEETS_KPI} / {RECOMMENDED_FOR_PRODUCTION}")
print(f"Reproduced holdout AUC (Notebook 36)      : {NB36_SUMMARY['reproduced_holdout_auc']:.4f} "
      f"(full history: {FULL_HISTORY_AUC:.4f})")
print(f"Real holdout population (Notebook 35/36)  : {N_HOLDOUT:,} customers")
print(f"Real overall default rate (Notebook 02)   : {LIVE_DEFAULT_RATE:.4%}")
print(f"EAD per account (Notebook 08, inherited)  : ${EAD_PER_ACCOUNT_USD:,}")
print(f"LGD assumption (Notebook 08, inherited)   : {LGD_ASSUMPTION:.0%}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

missing = []
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
except ImportError:
    missing.append("python-docx")
try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.worksheet.table import Table, TableStyleInfo
    from openpyxl.chart import BarChart, Reference
except ImportError:
    missing.append("openpyxl")
if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) +
                       "\nFix: pip install " + " ".join(missing))

print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: FINANCIAL PLANNING ASSUMPTIONS (EXPLICIT, EDITABLE)
# =============================================================================
_section("SECTION 3: Financial Planning Assumptions (Explicit, Editable)")

# --- This dataset has no real collections-intervention outcomes or project-cost data.
#     Every ASSUMPTION-labeled figure below is stated and editable -- nothing here is
#     fabricated as if it were measured. EAD/LGD are real inherited values (read
#     programmatically from Problem 1's Notebook 08), not re-guessed here. ---
FINANCIAL_ASSUMPTIONS = {
    "ead_per_account_usd": {"value": EAD_PER_ACCOUNT_USD, "source": "Notebook 08 (inherited, real value read programmatically)"},
    "lgd_assumption": {"value": LGD_ASSUMPTION, "source": "Notebook 08 (inherited, real value read programmatically)"},
    "early_intervention_success_rate": {
        "value": 0.20,
        "source": "ASSUMPTION -- illustrative efficacy of an early-warning intervention (credit-line "
                   "reduction, payment-plan offer, outreach call) when applied this many months earlier "
                   "than a full-history model would have flagged the same account; edit to your "
                   "institution's own collections-outcome data.",
    },
    "implementation_cost_usd": {
        "value": 55_000,
        "source": "ASSUMPTION -- illustrative one-time build/validate/deploy cost for the early-warning "
                   "scoring service (data science + risk review time); edit to your institution's actual "
                   "project cost.",
    },
    "annual_application_cycles": {
        "value": 4,
        "source": "ASSUMPTION -- how many times per year this scoring pipeline is re-run against a "
                   "holdout-sized population (illustrative quarterly cadence); edit to your institution's "
                   "actual monitoring frequency.",
    },
}
INTERVENTION_SUCCESS_RATE = FINANCIAL_ASSUMPTIONS["early_intervention_success_rate"]["value"]
IMPLEMENTATION_COST_USD = FINANCIAL_ASSUMPTIONS["implementation_cost_usd"]["value"]
ANNUAL_APPLICATION_CYCLES = FINANCIAL_ASSUMPTIONS["annual_application_cycles"]["value"]

assumptions_path = PILLAR_DIRS["p5_reporting_packaging"] / "financial_assumptions.json"
with open(assumptions_path, "w", encoding="utf-8") as f:
    json.dump(FINANCIAL_ASSUMPTIONS, f, indent=2)
for _k, _v in FINANCIAL_ASSUMPTIONS.items():
    print(f"  {_k}: {_v['value']}  ({_v['source']})")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: REAL EARLY-WARNING VALUE -- TIME GAINED & POPULATION FLAGGED
# =============================================================================
_section("SECTION 4: Real Early-Warning Value -- Time Gained & Population Flagged")

# --- MONTHS_EARLIER is a real, computed fact: the platform's real median
#     statement count (Notebook 34) minus the winning early window (Notebook
#     36) -- how many fewer monthly statements this model needs before it can
#     flag a typical customer, compared to waiting for their full history. ---
MEDIAN_STATEMENTS = STATEMENT_COUNT_STATS["median"]
MONTHS_EARLIER = MEDIAN_STATEMENTS - WINNING_K

# --- N_HOLDOUT_DEFAULTERS is a derived estimate, not a re-measured exact
#     count: Notebook 35/36 do not persist per-customer holdout labels, only
#     aggregate metrics. Notebook 02's split was stratified by target, so the
#     holdout's real default rate matches the platform's real overall
#     live_default_rate to within stratification rounding -- this is stated
#     explicitly as a derived figure, not presented as a separately measured one. ---
N_HOLDOUT_DEFAULTERS = round(N_HOLDOUT * LIVE_DEFAULT_RATE)
N_FLAGGED_TOP4PCT = round(0.04 * N_HOLDOUT)
HOLDOUT_TOP4PCT_CAPTURE = WINNING_K_RESULT["holdout_top4pct_capture"]
N_DEFAULTERS_CAPTURED_TOP4PCT = round(HOLDOUT_TOP4PCT_CAPTURE * N_HOLDOUT_DEFAULTERS)

print(f"Real median statement count, full history (Notebook 34)  : {MEDIAN_STATEMENTS:.0f}")
print(f"Winning early window (Notebook 36)                        : K={WINNING_K}")
print(f"Months of early warning gained (real, computed)           : {MONTHS_EARLIER:.0f}")
print(f"Real holdout population (Notebook 35/36)                  : {N_HOLDOUT:,}")
print(f"Estimated holdout defaulters (derived, see note above)    : {N_HOLDOUT_DEFAULTERS:,}")
print(f"Accounts flagged in the top 4% highest risk (real, exact 4% of {N_HOLDOUT:,}): {N_FLAGGED_TOP4PCT:,}")
print(f"Real top-4% capture rate (Notebook 35/36)                 : {HOLDOUT_TOP4PCT_CAPTURE:.1%}")
print(f"Estimated defaulters captured in that top-4% flag list    : {N_DEFAULTERS_CAPTURED_TOP4PCT:,}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: LOSS-PREVENTION OPPORTUNITY -- EARLY INTERVENTION ON TOP-4%-FLAGGED ACCOUNTS
# =============================================================================
_section("SECTION 5: Loss-Prevention Opportunity -- Early Intervention on Top-4%-Flagged Accounts")

PREVENTABLE_DEFAULTS = round(N_DEFAULTERS_CAPTURED_TOP4PCT * INTERVENTION_SUCCESS_RATE)
LOSS_PREVENTED_USD = PREVENTABLE_DEFAULTS * EAD_PER_ACCOUNT_USD * LGD_ASSUMPTION

print(f"Defaulters captured in the top-4% early flag list (derived)  : {N_DEFAULTERS_CAPTURED_TOP4PCT:,}")
print(f"ASSUMPTION early-intervention success rate                    : {INTERVENTION_SUCCESS_RATE:.0%}")
print(f"Estimated preventable defaults                                : {PREVENTABLE_DEFAULTS:,}")
print(f"Estimated loss prevented (this holdout sample, per cycle)     : ${LOSS_PREVENTED_USD:,.0f}")
print(f"Each prevented default is also flagged ~{MONTHS_EARLIER:.0f} months earlier than a "
      f"full-history-only model would have flagged it -- more runway per intervention, not just more of them.")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: ROI, INVESTMENT & PAYBACK PERIOD
# =============================================================================
_section("SECTION 6: ROI, Investment & Payback Period")

ANNUAL_BENEFIT_USD = LOSS_PREVENTED_USD * ANNUAL_APPLICATION_CYCLES
ROI_PCT = ((ANNUAL_BENEFIT_USD - IMPLEMENTATION_COST_USD) / IMPLEMENTATION_COST_USD) * 100 if IMPLEMENTATION_COST_USD else None
PAYBACK_MONTHS = (IMPLEMENTATION_COST_USD / (ANNUAL_BENEFIT_USD / 12)) if ANNUAL_BENEFIT_USD > 0 else None
# Honest fallback text/values for the (rare, small-population) case where the estimated
# annual benefit is $0 -- e.g. zero defaulters land in the top-4% flag list. Rather than
# crashing on a None format, or fabricating a fake number, every downstream consumer
# (narrative text, Word report, Excel, JSON summary, HTML dashboard) uses these safe
# strings/values so the "no measurable benefit under current assumptions" case is stated
# plainly wherever ROI/payback would otherwise appear.
ROI_DISPLAY = f"{ROI_PCT:,.0f}%" if ROI_PCT is not None else "N/A"
PAYBACK_DISPLAY = f"{PAYBACK_MONTHS:.1f} months" if PAYBACK_MONTHS else "N/A (no measurable benefit under current assumptions)"
PAYBACK_MONTHS_JSON = round(PAYBACK_MONTHS, 2) if PAYBACK_MONTHS else None
ROI_PCT_JSON = round(ROI_PCT, 1) if ROI_PCT is not None else None

print(f"Amount invested (ASSUMPTION, one-time)         : ${IMPLEMENTATION_COST_USD:,.0f}")
print(f"Estimated annual loss-prevention benefit        : ${ANNUAL_BENEFIT_USD:,.0f} "
      f"(= per-cycle benefit x {ANNUAL_APPLICATION_CYCLES} cycles/year, ASSUMPTION)")
print(f"Estimated ROI (Year 1)                          : {ROI_DISPLAY}")
print(f"Estimated payback period                        : {PAYBACK_DISPLAY}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: SMART SUGGESTIONS -- BOTTOM TO TOP MANAGEMENT
# =============================================================================
_section("SECTION 7: SMART Suggestions -- Bottom to Top Management")

SMART_SUGGESTIONS = [
    {"org_level": "Early-Warning Ops / Frontline Risk Analysts",
     "suggestion": f"Work the top-4% highest-risk flag list from the K={WINNING_K} model daily -- it needs "
                   f"only a customer's first {WINNING_K} statements, so accounts can be worked "
                   f"~{MONTHS_EARLIER:.0f} months sooner than waiting for full history. Target: contact "
                   f"attempt within 48 hours of a new top-4% flag."},
    {"org_level": "Collections/Risk Team Lead",
     "suggestion": f"Track the {PREVENTABLE_DEFAULTS:,}-account early-intervention goal (from the "
                   f"{INTERVENTION_SUCCESS_RATE:.0%} ASSUMPTION success rate) as a weekly KPI; report "
                   f"actual prevented-default count vs. this target each month to calibrate the assumption."},
    {"org_level": "Risk / Credit Analyst",
     "suggestion": f"Re-run this early-window scoring each time new statement data lands; monitor the "
                   f"split-half score PSI (Notebook 36 measured {NB36_SUMMARY['split_half_score_psi']:.4f}) "
                   f"and the bootstrap AUC CI (last measured "
                   f"[{NB36_SUMMARY['bootstrap_auc_ci'][0]:.4f}, {NB36_SUMMARY['bootstrap_auc_ci'][1]:.4f}]) "
                   f"-- re-validate if either drifts materially."},
    {"org_level": "Model Risk / Compliance (SR 11-7)",
     "suggestion": f"File Notebook 36's bootstrap CI, calibration gap ({NB36_SUMMARY['mean_calibration_gap']:.4f}), "
                   f"and AUC-retention KPI result ({'MET' if MEETS_KPI else 'NOT MET'}) with the model's annual "
                   f"validation packet; this model is currently "
                   f"{'RECOMMENDED' if RECOMMENDED_FOR_PRODUCTION else 'NOT RECOMMENDED'} for production use."},
    {"org_level": "Finance / Provisioning Team",
     "suggestion": f"Use the early-flagged population to time reserve builds ~{MONTHS_EARLIER:.0f} months "
                   f"sooner than the full-history model would allow -- coordinate with Problem 4's "
                   f"tier-differentiated LGD work for the $ reserve amount per flagged account."},
    {"org_level": "CFO / Executive Leadership",
     "suggestion": f"Approve the ${IMPLEMENTATION_COST_USD:,.0f} investment given an estimated "
                   f"{PAYBACK_DISPLAY} payback and {ROI_DISPLAY} Year-1 ROI from early "
                   f"intervention alone; revisit the {ANNUAL_APPLICATION_CYCLES}x/year cadence ASSUMPTION "
                   f"at the next quarterly business review."},
]
smart_df = pd.DataFrame(SMART_SUGGESTIONS)
smart_path = PILLAR_DIRS["p5_reporting_packaging"] / "p5_smart_suggestions.csv"
smart_df.to_csv(smart_path, index=False)
for _row in SMART_SUGGESTIONS:
    print(f"[{_row['org_level']}]\n  {_row['suggestion']}\n")
print(f"\u2705 Saved -> {smart_path.name}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: INLINE CHARTS
# =============================================================================
_section("SECTION 8: Inline Charts")

VIZ = {"ink": "#0B1F3A", "accent": "#C41E3A", "muted": "#8A93A6", "gold": "#C9A227", "surface": "#FFFFFF"}

fig1, ax1 = plt.subplots(figsize=(7, 5), dpi=150)
_bars = ax1.barh(["Full-history model\n(waits for all statements)", f"Early-window model\n(K={WINNING_K} statements)"],
                  [MEDIAN_STATEMENTS, WINNING_K], color=[VIZ["muted"], VIZ["accent"]])
for _b, _v in zip(_bars, [MEDIAN_STATEMENTS, WINNING_K]):
    ax1.text(_v, _b.get_y() + _b.get_height() / 2, f"  {_v:.0f} statements", va="center", fontsize=10)
ax1.set_xlabel("Statements needed before a risk score can be issued")
ax1.set_title(f"Problem 5: {MONTHS_EARLIER:.0f} Months of Early Warning Gained")
fig1.tight_layout()
chart1_path = PILLAR_DIRS["p5_reporting_packaging"] / "early_warning_time_gained_chart.png"
fig1.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.close(fig1)

fig2, ax2 = plt.subplots(figsize=(7, 5), dpi=150)
_bars2 = ax2.bar(["Total estimated\nholdout defaulters", "Captured in the\ntop-4% early flag list"],
                  [N_HOLDOUT_DEFAULTERS, N_DEFAULTERS_CAPTURED_TOP4PCT], color=[VIZ["muted"], VIZ["accent"]])
for _b, _v in zip(_bars2, [N_HOLDOUT_DEFAULTERS, N_DEFAULTERS_CAPTURED_TOP4PCT]):
    ax2.text(_b.get_x() + _b.get_width() / 2, _v, f"{_v:,}", ha="center", va="bottom", fontsize=10)
ax2.set_ylabel("Real holdout customers (estimated defaulter count)")
ax2.set_title(f"Problem 5: Defaulters Captured Early (K={WINNING_K}, top-4% flag list)")
fig2.tight_layout()
chart2_path = PILLAR_DIRS["p5_reporting_packaging"] / "defaulters_captured_early_chart.png"
fig2.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
plt.close(fig2)

print(f"\u2705 Saved -> {chart1_path.name}, {chart2_path.name}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: WORD REPORT -- Financial_Impact_Report.docx
# =============================================================================
_section("SECTION 9: Word Report -- Financial_Impact_Report.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = "" if v is None else str(v)
    return table


def _add_table_from_df(doc, df, max_rows=30):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    return table


doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 2, Problem 5: Early Payment Default Detection -- Financial Impact Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(doc, "1. Executive Summary", level=1)
doc.add_paragraph(
    f"The early-window model (K={WINNING_K}, {CHAMPION_NAME}) validated in Notebooks 34-36 needs only a "
    f"customer's first {WINNING_K} monthly statements to flag risk, versus the {MEDIAN_STATEMENTS:.0f} "
    f"statements a typical customer has by the time the full-history model can score them -- "
    f"{MONTHS_EARLIER:.0f} months of early warning gained, at {WINNING_K_RESULT['auc_retention_pct_of_full_history']:.1f}% "
    f"of the full-history model's real AUC. This model is currently "
    f"{'RECOMMENDED' if RECOMMENDED_FOR_PRODUCTION else 'NOT RECOMMENDED'} for production. Applying it to "
    f"the top-4% highest-risk accounts on the real {N_HOLDOUT:,}-customer holdout, an estimated "
    f"{N_DEFAULTERS_CAPTURED_TOP4PCT:,} real defaulters are captured early; at an ASSUMPTION "
    f"{INTERVENTION_SUCCESS_RATE:.0%} intervention success rate, this is estimated to prevent "
    f"${LOSS_PREVENTED_USD:,.0f} of loss per scoring cycle, for an estimated {PAYBACK_DISPLAY} "
    f"payback on a ${IMPLEMENTATION_COST_USD:,.0f} implementation investment."
)

_add_heading(doc, "2. Early-Warning Value: Time Gained & Population Flagged", level=1)
_add_kv_table(doc, {
    "median_statements_full_history": f"{MEDIAN_STATEMENTS:.0f}",
    "winning_early_window_k": WINNING_K,
    "months_earlier": f"{MONTHS_EARLIER:.0f}",
    "real_holdout_population": f"{N_HOLDOUT:,}",
    "estimated_holdout_defaulters": f"{N_HOLDOUT_DEFAULTERS:,}",
    "accounts_flagged_top_4pct": f"{N_FLAGGED_TOP4PCT:,}",
    "real_top_4pct_capture_rate": f"{HOLDOUT_TOP4PCT_CAPTURE:.1%}",
    "defaulters_captured_top_4pct": f"{N_DEFAULTERS_CAPTURED_TOP4PCT:,}",
})

_add_heading(doc, "3. Loss-Prevention Opportunity (Early Intervention)", level=1)
_add_kv_table(doc, {"defaulters_captured_early": N_DEFAULTERS_CAPTURED_TOP4PCT,
                     "intervention_success_rate_assumption": f"{INTERVENTION_SUCCESS_RATE:.0%}",
                     "preventable_defaults": PREVENTABLE_DEFAULTS,
                     "loss_prevented_per_cycle_usd": f"${LOSS_PREVENTED_USD:,.0f}"})

_add_heading(doc, "4. ROI, Investment & Payback", level=1)
_add_kv_table(doc, {"amount_invested_usd": f"${IMPLEMENTATION_COST_USD:,.0f}",
                     "annual_benefit_usd": f"${ANNUAL_BENEFIT_USD:,.0f}",
                     "roi_year_1_pct": ROI_DISPLAY, "payback_period_months": PAYBACK_DISPLAY})

_add_heading(doc, "5. SMART Suggestions by Organizational Level", level=1)
_add_table_from_df(doc, smart_df)

_add_heading(doc, "6. Assumptions & Sources", level=1)
_assump_df = pd.DataFrame([{"assumption": k, "value": v["value"], "source": v["source"]}
                            for k, v in FINANCIAL_ASSUMPTIONS.items()])
_add_table_from_df(doc, _assump_df)

report_path = PILLAR_DIRS["p5_reporting_packaging"] / "Financial_Impact_Report.docx"
doc.save(str(report_path))
print(f"\u2705 Saved -> {report_path.name}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: EXCEL WORKBOOK -- COLORFUL, TABLE + AUTOFILTER + CONDITIONAL FORMATTING + CHART
# =============================================================================
_section("SECTION 10: Excel Workbook -- Colorful, Table + AutoFilter + Conditional Formatting + Chart")

INK = "0B1F3A"
ACCENT = "C41E3A"
GOLD = "C9A227"
LIGHT = "F2F4F8"
WHITE = "FFFFFF"
USD_FMT = '$#,##0;($#,##0);-'

_assump_rows = {k: 2 + i for i, k in enumerate(FINANCIAL_ASSUMPTIONS.keys())}

wb = openpyxl.Workbook()

# --- Sheet 1: Assumptions (built first -- every formula below references these cells) ---
ws_assump = wb.active
ws_assump.title = "Assumptions"
ws_assump.append(["Assumption", "Value", "Source / Rationale"])
for _k, _v in FINANCIAL_ASSUMPTIONS.items():
    ws_assump.append([_k.replace("_", " ").title(), _v["value"], _v["source"]])
for _r in range(2, ws_assump.max_row + 1):
    ws_assump[f"B{_r}"].fill = PatternFill("solid", fgColor="FFFF00")
    ws_assump[f"B{_r}"].font = Font(name="Calibri", color="0000FF")
    ws_assump[f"C{_r}"].alignment = Alignment(wrap_text=True, vertical="top")
ws_assump[f"B{_assump_rows['lgd_assumption']}"].number_format = "0.0%"
ws_assump[f"B{_assump_rows['early_intervention_success_rate']}"].number_format = "0.0%"
ws_assump[f"B{_assump_rows['ead_per_account_usd']}"].number_format = USD_FMT
ws_assump[f"B{_assump_rows['implementation_cost_usd']}"].number_format = USD_FMT
ws_assump.column_dimensions["A"].width = 32
ws_assump.column_dimensions["B"].width = 14
ws_assump.column_dimensions["C"].width = 90
_tbl_assump = Table(displayName="Assumptions", ref=f"A1:C{ws_assump.max_row}")
_tbl_assump.tableStyleInfo = TableStyleInfo(name="TableStyleMedium4", showRowStripes=True)
ws_assump.add_table(_tbl_assump)

_ead_ref = f"Assumptions!$B${_assump_rows['ead_per_account_usd']}"
_lgd_ref = f"Assumptions!$B${_assump_rows['lgd_assumption']}"
_intervention_rate_ref = f"Assumptions!$B${_assump_rows['early_intervention_success_rate']}"
_cost_ref = f"Assumptions!$B${_assump_rows['implementation_cost_usd']}"
_cycles_ref = f"Assumptions!$B${_assump_rows['annual_application_cycles']}"

# --- Sheet 2: Early-Warning Impact -- key figures as REAL FORMULAS referencing Assumptions,
#     so editing an assumption recalculates every dollar figure. ---
ws_impact = wb.create_sheet("Early-Warning Impact")
ws_impact.append(["Metric", "Value"])
_impact_rows_static = [
    ("Median Statements, Full History (Notebook 34)", MEDIAN_STATEMENTS),
    ("Winning Early Window K (Notebook 36)", WINNING_K),
    ("Months Earlier (real, computed)", MONTHS_EARLIER),
    ("Real Holdout Population", N_HOLDOUT),
    ("Estimated Holdout Defaulters", N_HOLDOUT_DEFAULTERS),
    ("Real Top-4% Capture Rate", HOLDOUT_TOP4PCT_CAPTURE),
    ("Defaulters Captured (Top-4% Flag List)", N_DEFAULTERS_CAPTURED_TOP4PCT),
]
for _label, _val in _impact_rows_static:
    ws_impact.append([_label, _val])
_preventable_row = ws_impact.max_row + 1
ws_impact.append(["Preventable Defaults", f"=ROUND(B7*{_intervention_rate_ref},0)"])
_loss_prevented_row = ws_impact.max_row + 1
ws_impact.append(["Loss Prevented / Cycle (USD)", f"=B{_preventable_row}*{_ead_ref}*{_lgd_ref}"])
_annual_benefit_row = ws_impact.max_row + 1
ws_impact.append(["Annual Benefit (USD)", f"=B{_loss_prevented_row}*{_cycles_ref}"])
ws_impact["B6"].number_format = "0.0%"
ws_impact[f"B{_loss_prevented_row}"].number_format = USD_FMT
ws_impact[f"B{_annual_benefit_row}"].number_format = USD_FMT
ws_impact.column_dimensions["A"].width = 42
ws_impact.column_dimensions["B"].width = 20
_tbl_impact = Table(displayName="EarlyWarningImpact", ref=f"A1:B{ws_impact.max_row}")
_tbl_impact.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
ws_impact.add_table(_tbl_impact)

_chart = BarChart()
_chart.title = "Defaulters: Total Estimated vs. Captured Early"
_chart.y_axis.title = "Customers"
_cat_row_start, _cat_row_end = 6, 8  # "Estimated Holdout Defaulters" & "Defaulters Captured..." rows
_data = Reference(ws_impact, min_col=2, min_row=1, max_row=ws_impact.max_row)
_cats = Reference(ws_impact, min_col=1, min_row=2, max_row=ws_impact.max_row)
_chart.add_data(_data, titles_from_data=True)
_chart.set_categories(_cats)
_chart.width, _chart.height = 20, 10
ws_impact.add_chart(_chart, "D2")

# --- Sheet 3: SMART Suggestions (real Excel Table -> native AutoFilter dropdowns) ---
ws_smart = wb.create_sheet("SMART Suggestions")
ws_smart.append(["Org Level", "Suggestion"])
for _row_data in SMART_SUGGESTIONS:
    ws_smart.append([_row_data["org_level"], _row_data["suggestion"]])
_last_row_smart = ws_smart.max_row
_tbl_smart = Table(displayName="SmartSuggestions", ref=f"A1:B{_last_row_smart}")
_tbl_smart.tableStyleInfo = TableStyleInfo(name="TableStyleMedium7", showRowStripes=True)
ws_smart.add_table(_tbl_smart)
ws_smart.column_dimensions["A"].width = 34
ws_smart.column_dimensions["B"].width = 100
for _r in range(2, _last_row_smart + 1):
    ws_smart[f"B{_r}"].alignment = Alignment(wrap_text=True, vertical="top")

# --- Sheet 4: Executive Summary (KPI cards), inserted first, populated last ---
ws_exec = wb.create_sheet("Executive Summary", 0)
wb.active = 0
ws_exec.sheet_view.showGridLines = False
ws_exec["B2"] = "AMEX RiskIQ -- Problem 5: Early Payment Default Detection"
ws_exec["B2"].font = Font(name="Calibri", size=16, bold=True, color=WHITE)
ws_exec["B2"].fill = PatternFill("solid", fgColor=INK)
ws_exec.merge_cells("B2:F2")
ws_exec["B3"] = "Financial Impact Summary"
ws_exec["B3"].font = Font(name="Calibri", size=11, italic=True, color=INK)
ws_exec.merge_cells("B3:F3")

_kpi_rows = [
    ("Months of Early Warning Gained", f"{MONTHS_EARLIER:.0f} months  (reported, see Early-Warning Impact sheet)", False, LIGHT),
    ("Defaulters Captured Early", "='Early-Warning Impact'!B8", True, LIGHT),
    ("Est. Loss Prevented / Cycle", f"='Early-Warning Impact'!B{_loss_prevented_row}", True, GOLD),
    ("Amount Invested", f"=\"$\"&TEXT({_cost_ref},\"#,##0\")", True, LIGHT),
    ("Estimated Year-1 ROI", f"{ROI_DISPLAY}  (reported, see Section 6)", False, ACCENT),
    ("Estimated Payback", f"{PAYBACK_DISPLAY}  (reported, see Section 6)", False, ACCENT),
    ("Deployment Status", f"{'RECOMMENDED' if RECOMMENDED_FOR_PRODUCTION else 'NOT RECOMMENDED'} for production", False, ACCENT if not RECOMMENDED_FOR_PRODUCTION else "63BE7B"),
]
_row = 5
for _label, _value, _is_formula, _fill in _kpi_rows:
    ws_exec.cell(row=_row, column=2, value=_label).font = Font(name="Calibri", size=11, color=INK)
    _cell = ws_exec.cell(row=_row, column=4, value=_value)
    _cell.font = Font(name="Calibri", size=13, bold=True, color=(WHITE if _fill in (GOLD, ACCENT) else INK))
    _cell.fill = PatternFill("solid", fgColor=_fill)
    _cell.alignment = Alignment(horizontal="center", wrap_text=not _is_formula)
    if _is_formula and _label in ("Defaulters Captured Early",):
        _cell.number_format = "#,##0"
    elif _is_formula and "Loss Prevented" in _label:
        _cell.number_format = USD_FMT
    ws_exec.merge_cells(start_row=_row, start_column=4, end_row=_row, end_column=5)
    _row += 1
ws_exec["B14"] = "Rows 6-8 recalculate live from the Assumptions and Early-Warning Impact sheets."
ws_exec["B14"].font = Font(name="Calibri", size=9, italic=True, color="8A93A6")
ws_exec.merge_cells("B14:F14")
for _col, _w in zip("BCDEF", [32, 3, 22, 22, 3]):
    ws_exec.column_dimensions[_col].width = _w

for _ws in (ws_impact, ws_smart, ws_assump):
    for _cell in _ws[1]:
        _cell.font = Font(name="Calibri", bold=True, color=WHITE)
        _cell.fill = PatternFill("solid", fgColor=INK)

workbook_path = PILLAR_DIRS["p5_reporting_packaging"] / "AMEX_Problem5_Financial_Impact_Workbook.xlsx"
wb.save(str(workbook_path))
print(f"\u2705 Saved -> {workbook_path.name}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: INTERACTIVE HTML DASHBOARD
# =============================================================================
_section("SECTION 11: Interactive HTML Dashboard")

_smart_json = json.dumps(SMART_SUGGESTIONS)
_org_levels = sorted({r["org_level"] for r in SMART_SUGGESTIONS})

_html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Problem 5 -- Financial Impact Dashboard</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4"></script>
<style>
  :root { --ink:#0B1F3A; --accent:#C41E3A; --gold:#C9A227; --muted:#8A93A6; --bg:#F2F4F8; --card:#FFFFFF; }
  body { font-family: Calibri, Arial, sans-serif; background: var(--bg); color: var(--ink); margin: 0; padding: 24px; }
  h1 { font-size: 22px; margin-bottom: 4px; }
  .sub { color: var(--muted); margin-bottom: 20px; }
  .kpi-row { display: flex; flex-wrap: wrap; gap: 14px; margin-bottom: 24px; }
  .kpi { background: var(--card); border-radius: 10px; padding: 16px 20px; box-shadow: 0 1px 3px rgba(0,0,0,.12); min-width: 190px; flex: 1; }
  .kpi .label { font-size: 12px; color: var(--muted); text-transform: uppercase; }
  .kpi .value { font-size: 22px; font-weight: 700; margin-top: 4px; }
  .panel { background: var(--card); border-radius: 10px; padding: 18px; margin-bottom: 20px; box-shadow: 0 1px 3px rgba(0,0,0,.12); }
  table { width: 100%; border-collapse: collapse; font-size: 13px; }
  th, td { text-align: left; padding: 8px 10px; border-bottom: 1px solid #E4E7EE; }
  th { background: var(--ink); color: #fff; }
  select { padding: 6px 10px; border-radius: 6px; border: 1px solid var(--muted); font-size: 13px; margin-bottom: 12px; }
  canvas { max-height: 360px; }
  .badge { display: inline-block; padding: 3px 10px; border-radius: 12px; font-size: 12px; font-weight: 700; }
</style>
</head>
<body>
<h1>AMEX RiskIQ -- Problem 5: Early Payment Default Detection</h1>
<div class="sub">Financial Impact Dashboard -- real Notebook 34-36 results, ASSUMPTION values clearly marked</div>

<div class="kpi-row">
  <div class="kpi"><div class="label">Months Early Warning Gained</div><div class="value">__MONTHS_EARLIER__</div></div>
  <div class="kpi"><div class="label">Defaulters Captured Early</div><div class="value">__DEFAULTERS_CAPTURED__</div></div>
  <div class="kpi"><div class="label">Est. Loss Prevented / Cycle</div><div class="value">__LOSS_PREVENTED__</div></div>
  <div class="kpi"><div class="label">Est. Year-1 ROI</div><div class="value">__ROI__</div></div>
  <div class="kpi"><div class="label">Est. Payback</div><div class="value">__PAYBACK__</div></div>
  <div class="kpi"><div class="label">Deployment Status</div><div class="value"><span class="badge" style="background:__STATUS_COLOR__;color:#fff;">__STATUS__</span></div></div>
</div>

<div class="panel">
  <canvas id="captureChart"></canvas>
</div>

<div class="panel">
  <label for="orgFilter"><b>SMART Suggestions -- filter by organizational level</b></label><br/>
  <select id="orgFilter"></select>
  <table id="smartTable"><thead><tr><th>Org Level</th><th>Suggestion</th></tr></thead><tbody></tbody></table>
</div>

<script>
const smartData = __SMART_JSON__;
const orgLevels = __ORG_LEVELS__;

const ctx = document.getElementById("captureChart").getContext("2d");
new Chart(ctx, {
  type: "bar",
  data: {
    labels: ["Estimated Holdout Defaulters", "Captured in Top-4% Early Flag List"],
    datasets: [{ data: [__N_HOLDOUT_DEFAULTERS__, __N_CAPTURED__], backgroundColor: ["#8A93A6", "#C41E3A"] }],
  },
  options: { responsive: true, plugins: { legend: { display: false } },
             scales: { y: { ticks: { callback: v => v.toLocaleString() } } } },
});

function renderSmart(filterLevel) {
  const tbody = document.querySelector("#smartTable tbody");
  tbody.innerHTML = "";
  smartData.filter(r => filterLevel === "All" || r.org_level === filterLevel).forEach(r => {
    const tr = document.createElement("tr");
    tr.innerHTML = `<td>${r.org_level}</td><td>${r.suggestion}</td>`;
    tbody.appendChild(tr);
  });
}

const orgSelect = document.getElementById("orgFilter");
["All", ...orgLevels].forEach(level => {
  const opt = document.createElement("option");
  opt.value = level; opt.textContent = level;
  orgSelect.appendChild(opt);
});
orgSelect.onchange = () => renderSmart(orgSelect.value);
renderSmart("All");
</script>
</body>
</html>
"""
_html = (_html
         .replace("__MONTHS_EARLIER__", f"{MONTHS_EARLIER:.0f}")
         .replace("__DEFAULTERS_CAPTURED__", f"{N_DEFAULTERS_CAPTURED_TOP4PCT:,}")
         .replace("__LOSS_PREVENTED__", f"${LOSS_PREVENTED_USD:,.0f}")
         .replace("__ROI__", ROI_DISPLAY)
         .replace("__PAYBACK__", PAYBACK_DISPLAY)
         .replace("__STATUS__", "RECOMMENDED" if RECOMMENDED_FOR_PRODUCTION else "NOT RECOMMENDED")
         .replace("__STATUS_COLOR__", "#16a34a" if RECOMMENDED_FOR_PRODUCTION else "#dc2626")
         .replace("__N_HOLDOUT_DEFAULTERS__", str(N_HOLDOUT_DEFAULTERS))
         .replace("__N_CAPTURED__", str(N_DEFAULTERS_CAPTURED_TOP4PCT))
         .replace("__SMART_JSON__", _smart_json)
         .replace("__ORG_LEVELS__", json.dumps(_org_levels)))

dashboard_path = PILLAR_DIRS["p5_reporting_packaging"] / "financial_impact_dashboard.html"
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(_html)
print(f"\u2705 Saved -> {dashboard_path.name}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: VERIFICATION
# =============================================================================
_section("SECTION 12: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Months earlier is a positive real number", MONTHS_EARLIER > 0, f"({MONTHS_EARLIER})")
_check("Defaulters captured does not exceed estimated holdout defaulters",
       N_DEFAULTERS_CAPTURED_TOP4PCT <= N_HOLDOUT_DEFAULTERS)
_check("Preventable defaults does not exceed defaulters captured",
       PREVENTABLE_DEFAULTS <= N_DEFAULTERS_CAPTURED_TOP4PCT)
_check("ROI is a finite number (implementation cost is a fixed, non-zero assumption)",
       ROI_PCT is not None)
_check("Payback is a positive finite number when there is measurable annual benefit, "
       "and explicitly undefined (None) otherwise -- never a crash or a fabricated value",
       (PAYBACK_MONTHS is not None and PAYBACK_MONTHS > 0) if ANNUAL_BENEFIT_USD > 0
       else PAYBACK_MONTHS is None)
_check("EAD/LGD were inherited from Notebook 08, not re-guessed",
       EAD_PER_ACCOUNT_USD == NB08_SUMMARY["ead_per_account_usd_assumption"]
       and LGD_ASSUMPTION == NB08_SUMMARY["lgd_assumption"])

_expected_files = [assumptions_path, smart_path, chart1_path, chart2_path, report_path,
                    workbook_path, dashboard_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 37 verification checks failed. See \u274c line above.")
print("\nAll Notebook 37 checks passed.")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: WRITE NOTEBOOK 37 SUMMARY -- PROBLEM 5 COMPLETE
# =============================================================================
_section("SECTION 13: Write Notebook 37 Summary -- Problem 5 Complete")

notebook_37_summary = {
    "notebook": "37_early_payment_default_reporting_packaging", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 5, "problem_name": "Early Payment Default Detection",
    "phase": "Phase 2 -- Regulatory & Loss Provisioning", "problem_5_complete": True,
    "winning_k": WINNING_K, "meets_kpi_target": MEETS_KPI, "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
    "months_earlier": MONTHS_EARLIER, "defaulters_captured_early": N_DEFAULTERS_CAPTURED_TOP4PCT,
    "loss_prevented_per_cycle_usd": round(LOSS_PREVENTED_USD, 2),
    "roi_year_1_pct": ROI_PCT_JSON, "payback_period_months": PAYBACK_MONTHS_JSON,
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb37_summary_path = ARTIFACTS_DIR / "notebook_37_summary.json"
with open(nb37_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_37_summary, f, indent=2)
print(f"\u2705 Saved -> {nb37_summary_path.name}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: COMPLETION SUMMARY -- PROBLEM 5 COMPLETE
# =============================================================================
_section("SECTION 14: Notebook 37 Complete -- Problem 5 Complete")

print("NOTEBOOK 37: FINANCIAL-IMPACT REPORTING & PACKAGING -- COMPLETE")
print("PROBLEM 5 (EARLY PAYMENT DEFAULT DETECTION) -- ALL 4 NOTEBOOKS COMPLETE")
print(f"  Winning window / meets KPI / recommended : K={WINNING_K} / {MEETS_KPI} / {RECOMMENDED_FOR_PRODUCTION}")
print(f"  Months of early warning gained             : {MONTHS_EARLIER:.0f}")
print(f"  Estimated loss prevented per cycle          : ${LOSS_PREVENTED_USD:,.0f}")
print(f"  Estimated Year-1 ROI / payback               : {ROI_DISPLAY} / {PAYBACK_DISPLAY}")
print(f"  Files produced                              : {len(_expected_files) + 1}")
for _p in _expected_files + [nb37_summary_path]:
    print(f"    - {_p.name}")
print("\n  PHASE 2 (Problems 3, 4, 5) is now complete. Next: push Phase 2 to GitHub and apply the same "
      "hardening pass (tests, shared/ additions, MODEL_CARD/CHANGELOG, CI) Phase 1 already has.")
print("\n\u2705 Ready to proceed.")
